In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.enviroment-config

In [0]:
%run ../00-common/04.gold_helpers

In [0]:
target_table=f"{catalog_name}.{gold_schema}.dim_constructors"

In [0]:
from pyspark.sql import functions as F

In [0]:
constructor_df=((spark.table(f"{catalog_name}.{silver_schema}.constructors")
            .filter(F.col("batch_id")==v_batch_id))
)

In [0]:
ref_nationality_region_df=spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")


In [0]:
dim_constructor_df=(
    constructor_df
    .join(ref_nationality_region_df,
          constructor_df.nationality==ref_nationality_region_df.nationality,
          how="left"
          )
    .select(
        constructor_df.constructor_id,
        constructor_df.constructor_name,
        constructor_df.nationality,
        ref_nationality_region_df.region.alias("nationality_region")
        )
       
)

In [0]:
display(dim_constructor_df)

In [0]:
# (
#     dim_constructor_df
#     .write
#     .format("delta")
#     .mode("overwrite")
#     .saveAsTable(target_table)
# )

In [0]:
write_to_gold(
    input_df=dim_constructor_df,
    target_table=target_table,
    merge_condition="t.constructor_id=s.constructor_id",
    columns_to_update=["constructor_name","nationality","nationality_region"] 
)

In [0]:
display(spark.table(target_table))